# 02aiii · Decoupler/PyDESeq2 pseudobulk DEG analysis, ILC3_NCRhi (reproductive tract-derived vs. non-reproductive tract-derived)

Marker identification with differential gene expression (DEG) analysis.

**Pipeline:** based on the [**Pseudobulk Enrichment Analysis** tutorial](https://decoupler.readthedocs.io/en/latest/notebooks/scell/rna_psbk.html) by `decoupler`.
1. Load annotated .h5ad object containing raw counts.
2. Subset AnnData to groups (typically cell types) of interest in order to perform targeted DEG analysis.
3. Pseudobulk the data by donor x group.
4. Filter out low-quality pseudobulk samples and lowly expressed genes.
5. Run DESeq2 to identify pairwise differentially expressed genes.
6. Plot top DEGs on a volcano plot.
7. Save all upregulated and downregulated DEGs.

## Configuration

In [ ]:
import os, sys

# ── Paths ─────────────────────────────────────────────────────────────────────
OBJECT          = "crossdisease_ILC"
MAIN_DIR        = "/nfs/team292/projects/PanTissue/"
OUTPUT_DIR      = os.path.join(MAIN_DIR, "results/temp/02_annotation/immune/acrosslifespan/noHormones/04_crossdisease_ILC/")
FIG_DIR         = os.path.join(OUTPUT_DIR, "figures/")
ANNOTATED_H5AD  = os.path.join(OUTPUT_DIR, f"annotated_{OBJECT}.h5ad")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

# ── Adjustable parameters ───────────────────────────────────────────────——————
DONOR_COL         = "Donor_id"
ANNOT_COL         = "celltype"
ANNOT             = ["ILC3_NCRhi"]
GROUPS_COL        = "TISSUE_BROAD"
GROUPS            = [
    "ReproTract",
    "NonReproTract",
]

MIN_CELLS  = 10
MIN_COUNTS = 1000

## Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import anndata as ad
import scanpy as sc
import pandas as pd
import numpy as np

import decoupler as dc
from pydeseq2.dds import DeseqDataSet, DefaultInference
from pydeseq2.ds import DeseqStats

import matplotlib.pyplot as plt

# 1. Load annotated object

In [ ]:
adata = ad.read_h5ad(ANNOTATED_H5AD)
adata

In [ ]:
mask = (
    # ~adata.obs["Organ"].isin([
    #     "Mesonephros"
    # ])
    # &
    ~adata.obs["Developmental_stage_v2"].isin([
        "Fetal",
        "Pediatric"
    ])
)

adata = adata[mask].copy()

In [ ]:
# —— Broad tissue labels ———————————————————————————————————————————————————————
tissue_broad = {
    "Ovary": "ReproTract",
    "Fallopian Tube": "ReproTract",
    "Cervix": "ReproTract",
    "Vagina": "ReproTract",
    
    "Endometrium": "ReproTract",
    "Myometrium": "ReproTract",
    "Whole Uterus": "ReproTract",
    "Menstrual fluid": "ReproTract",
    
    "Buccal": "NonReproTract",
    "Colon": "NonReproTract",
    "Gingiva": "NonReproTract",
    "Lung": "NonReproTract"
}

adata.obs["TISSUE_BROAD"] = adata.obs["TISSUE_MERGED"].map(tissue_broad)
# adata.obs["TISSUE_BROAD"].isna().sum()
adata.obs["TISSUE_BROAD"].value_counts()

In [ ]:
adata.obs[GROUPS_COL].value_counts()

In [ ]:
adata.obs["Donor_id"].value_counts()

In [ ]:
adata.raw is None

# 2. Subset to cells of interest

In [ ]:
print(f"Subsetting to only include {', '.join(ANNOT)}...")
adata_sub = adata[adata.obs[ANNOT_COL].isin(ANNOT)].copy()

print(f">> {adata_sub.n_obs} cells kept\n")
adata_sub.obs[GROUPS_COL].value_counts()

# 3. Pseudobulk counts

**Aggregate raw counts using `decoupler`:**

`pdata` is an AnnData object, where:
* Each row (`obs`) is one pseudobulk sample (donor x cell type)
* Each column (`var`) is a gene
* `pdata.obs` contains donor, cell type, and QC columns
    * `psbulk_cells`: number of cells aggregated
    * `psbulk_counts`: total counts in that pseudobulk sample

Note that `obs` columns are only carried over if it has the **same value** across the pseudobulk level (i.e., donor).

In [ ]:
# —— Aggregate counts per donor × cell type using decoupler ————————————
pdata = dc.pp.pseudobulk(
    adata       = adata_sub,
    sample_col  = DONOR_COL,   # column in adata.obs with donor/sample IDs
    groups_col  = ANNOT_COL,   # column in adata.obs with cell type labels
    mode        = "sum",
    skip_checks = True
)
pdata

The following plot shows each pseudobulk sample as a point, with number of cells on one axis and total counts on the other. Samples in the lower-left corner are likely low quality, and can be excluded from the analysis using `dc.pp.filter_samples()`.

Filter thresholds are always dataset-dependent and arbitrary, but a common guideline is to retain samples with at least **10 cells** and **1000 total counts.**

In [ ]:
# —— Visualise QC metrics before filtering —————————————————————————————
QC_METRICS = [DONOR_COL, ANNOT_COL]

for METRIC in QC_METRICS:
    # —— Conditionally suppress legend if too many group levels ————————
    n_levels = pdata.obs[METRIC].nunique()
    legend_setting = "auto" if n_levels <= 10 else False
    
    dc.pl.filter_samples(
        adata          = pdata,
        groupby        = [METRIC],
        min_cells      = MIN_CELLS,
        min_counts     = MIN_COUNTS,
        figsize        = (6,3),
        kw_scatterplot = {"legend": legend_setting}
        # kw_scatterplot passes extra keyword arguments directly to
        # seaborn.scatterplot, which draws the points in the QC plot
    )

In [ ]:
# —— Apply the filter ——————————————————————————————————————————————————
pre_filt_cells = pdata.n_obs
print(f"Removing samples with at least {MIN_CELLS} cells and {MIN_COUNTS} counts...")
dc.pp.filter_samples(
    pdata,
    min_cells  = MIN_CELLS,
    min_counts = MIN_COUNTS
)
post_filt_cells = pdata.n_obs
print(f">> {pre_filt_cells - post_filt_cells} pseudobulk samples removed.\n")
pdata

In [ ]:
donors = pdata.obs["Donor_id"].unique()
print(', '.join(donors))

# 4. Filter genes

Lowly expressed genes will also be filtered prior to downstream analysis. This step should be performed at the group (i.e., cell type) level, as different cell types may express distinct sets of genes. The following step uses the default parameters as specified in the `decoupler` Pseudobulk Enrichment Analysis tutorial.

The following function stores the filtered pseudobulk AnnDatas for each subset in a dictionary, keyed by subset name, so it is ready to pass directly into DESeq2 in the next step without recomputing.

In [ ]:
# —— Filter genes across the full pdata ————————————————————————————————
pdata_filtered = pdata.copy()

dc.pp.filter_by_expr(
    adata           = pdata_filtered,
    group           = GROUPS_COL,
    min_count       = 10,
    min_total_count = 15,
    large_n         = 10,
    min_prop        = 0.7
)

dc.pp.filter_by_prop(
    adata     = pdata_filtered,
    min_prop  = 0.1,
    min_smpls = 2
)

print(f"Genes remaining after filtering: {pdata_filtered.shape[1]}")

# 5. Run DESeq2

In this notebook, we run DEG analysis via **pairwise comparison**.

In [ ]:
# —— Define a reusable DEG filter function —————————————————————————————
def filter_degs(
    results_df,
    padj_thresh:  float=0.05,
    lfc_thresh:   float=1.0
):
    """
    Filters a PyDESeq2 results DataFrame to significant DEGs.

    Parameters
    ----------
    results_df : pd.DataFrame
        Output from DeseqStats.results_df
    padj_thresh : float
        Adjusted p-value threshold. Default 0.05
    lfc_thresh : float
        Absolute log2 fold change threshold. Default 1.0 (i.e. 2-fold)
    """
    mask = (
        (results_df["padj"] < padj_thresh) &
        (results_df["log2FoldChange"].abs() > lfc_thresh)
    )
    return results_df[mask].sort_values("padj")

In [ ]:
# —— Extract counts and metadata as DataFrames for PyDESeq2 ————————————
counts_df   = pd.DataFrame(
    pdata_filtered.X,
    index   = pdata_filtered.obs_names.astype(str),
    columns = pdata_filtered.var_names
)
counts_df = counts_df.round().astype(int)

metadata_df = pdata_filtered.obs[[GROUPS_COL]].copy()
metadata_df.index = metadata_df.index.astype(str)

# —— Initialise and fit DESeq2 —————————————————————————————————————————
inference = DefaultInference(n_cpus=8)

dds = DeseqDataSet(
    counts      = counts_df,
    metadata    = metadata_df,
    design      = f"~{GROUPS_COL}",
    refit_cooks = True,
    inference   = inference
)

dds.deseq2()

# —— Extract pairwise contrast and run Wald test ———————————————————————
stat_res = DeseqStats(
    dds,
    contrast  = [GROUPS_COL, GROUPS[0], GROUPS[1]],
    inference = inference
)
stat_res.summary()

results               = stat_res.results_df
results["comparison"] = f"{GROUPS[0]}_vs_{GROUPS[1]}"

# —— Filter to significant DEGs ————————————————————————————————————————
sig_results = filter_degs(
    results,
    padj_thresh = 0.05,
    lfc_thresh  = 1.0
)

print(f"\nSignificant DEGs ({GROUPS[0]} vs {GROUPS[1]}): {len(sig_results)}")

# 6. Plot DEGs

In [ ]:
# —— Plot volcano ——————————————————————————————————————————————————————
fig = dc.pl.volcano(
    data       = results,
    x          = "log2FoldChange",
    y          = "pvalue",
    top        = 10,
    figsize    = (5, 4),
    return_fig = True
)

fig.axes[0].set_title(
    f"{GROUPS[0]} vs {GROUPS[1]}",
    fontsize = 12,
    pad      = 10
)

fig.savefig(
    f"{FIG_DIR}volcano_01_PW_{GROUPS[0]}_vs_{GROUPS[1]}_{OBJECT}.png",
    bbox_inches = "tight"
)

plt.show()

# 7. Save DEGs

In [ ]:
# —— Build and split summary DEG table —————————————————————————————————
deg_summary_df = pd.DataFrame({
    "gene_name": sig_results.index,
    GROUPS_COL:  f"{GROUPS[0]}_vs_{GROUPS[1]}",
    "log2FC":    sig_results["log2FoldChange"].values,
    "padj":      sig_results["padj"].values
})

# —— Split into upregulated and downregulated —————————————————————————
deg_up   = deg_summary_df[deg_summary_df["log2FC"] > 0].reset_index(drop=True)
deg_down = deg_summary_df[deg_summary_df["log2FC"] < 0].reset_index(drop=True)

# —— Save to CSV ———————————————————————————————————————————————————————
deg_up.to_csv(
    f"{OUTPUT_DIR}DESeq2_upDEGs_PW_{GROUPS[0]}_vs_{GROUPS[1]}_{OBJECT}.csv",
    index = False
)
deg_down.to_csv(
    f"{OUTPUT_DIR}DESeq2_downDEGs_PW_{GROUPS[0]}_vs_{GROUPS[1]}_{OBJECT}.csv",
    index = False
)

# —— Print summary ————————————————————————————————————————————————————
print(f"Upregulated DEGs   ({GROUPS[0]} > {GROUPS[1]}): {len(deg_up)}")
print(f"Downregulated DEGs ({GROUPS[1]} > {GROUPS[0]}): {len(deg_down)}")

—— FIN ——